# Phase 4b — Hypothesis Tests + Classifier (n = 127, expanded sample)

Same structure as Phase 4a, pointed at the expanded 127-record dataset (the original 75
plus 52 newly collected, misdirection-domain records). This is the **replication** side
of the paper's central comparison — every number here should be compared directly
against Phase 4a's equivalent.

Fully independent of Phase 4a: rebuilds every feature from scratch, no shared state.

In [1]:
import numpy as np
import json
import os
from collections import Counter
import matplotlib.pyplot as plt
import seaborn as sns
from statsmodels.stats.multitest import multipletests
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import balanced_accuracy_score, f1_score, roc_auc_score

np.random.seed(42)

NPZ_PATH  = '/kaggle/input/notebooks/gunjalgupta/phase3-v2/pivot_research/phase3_attention_v3.npz'
META_PATH = '/kaggle/input/notebooks/gunjalgupta/phase3-v2/pivot_research/phase3_metadata_v3.jsonl'
OUTPUT_DIR = '/kaggle/working/pivot_research'
os.makedirs(OUTPUT_DIR, exist_ok=True)

N_LAYERS, N_HEADS, N_QUERIES = 28, 12, 30
EARLY_KEY_RANGE = 50
EARLY_LAYERS = list(range(0, N_LAYERS // 3))
LATE_LAYERS  = list(range(2 * N_LAYERS // 3, N_LAYERS))

## Load data

Same defensive pattern as Phase 4a: assert the sample size and label counts immediately,
so a wrong path fails here rather than downstream.

In [2]:
npz = np.load(NPZ_PATH)
with open(META_PATH) as f:
    metadata = [json.loads(l) for l in f if l.strip()]
n_records = len(metadata)

weights = np.stack([npz[f'weights_{i:03d}'] for i in range(n_records)], axis=0)
outputs = np.stack([npz[f'outputs_{i:03d}'] for i in range(n_records)], axis=0)
labels = np.array([m['label'] for m in metadata])
categories = np.array([m['category'] for m in metadata])
y = (labels == 'correction').astype(int)
valid_key_starts = np.array([m['valid_key_start'] for m in metadata])
prompt_ids = np.array([m['prompt_id'] for m in metadata])   # kept for Phase 4c cross-checks
corr_mask, halluc_mask = y == 1, y == 0

print(f"n={n_records}: correction={corr_mask.sum()}, hallucination_pivot={halluc_mask.sum()}")
assert n_records == 127 and corr_mask.sum() == 22, \
    f"Expected n=127, 22 correction -- got n={n_records}, {corr_mask.sum()} correction. Check NPZ_PATH."
print("Category distribution:", dict(Counter(categories)))

n=127: correction=22, hallucination_pivot=105
Category distribution: {np.str_('misdirection'): 67, np.str_('math'): 36, np.str_('logic'): 24}


## Shared utilities

Identical to Phase 4a — permutation test + FDR correction, applied per head.

In [3]:
def permutation_test(group1, group2, n_perm=2000, seed=42):
    rng = np.random.default_rng(seed)
    observed = group1.mean() - group2.mean()
    combined = np.concatenate([group1, group2])
    n1 = len(group1)
    diffs = np.zeros(n_perm)
    for i in range(n_perm):
        rng.shuffle(combined)
        diffs[i] = combined[:n1].mean() - combined[n1:].mean()
    return observed, (np.abs(diffs) >= np.abs(observed)).mean()


def run_per_head_test(features):
    p_values = np.zeros((N_LAYERS, N_HEADS))
    effect_sizes = np.zeros((N_LAYERS, N_HEADS))
    for L in range(N_LAYERS):
        for H in range(N_HEADS):
            diff, p = permutation_test(features[corr_mask, L, H], features[halluc_mask, L, H])
            p_values[L, H] = p
            effect_sizes[L, H] = diff
    reject, adj_p, _, _ = multipletests(p_values.flatten(), alpha=0.05, method='fdr_bh')
    adj_p = np.asarray(adj_p).reshape(N_LAYERS, N_HEADS)
    reject = np.asarray(reject).reshape(N_LAYERS, N_HEADS)
    return p_values, effect_sizes, adj_p, reject

## H1 — Early-Context Attention Mass

Same definition as Phase 4a — see that notebook for the full explanation. Feature:
attention mass on the first 50 real keys, averaged over the 30-query window.

In [4]:
def compute_h1(weights, valid_key_starts):
    h1 = np.zeros((n_records, N_LAYERS, N_HEADS))
    for i in range(n_records):
        v = valid_key_starts[i]
        early_end = min(v + EARLY_KEY_RANGE, weights.shape[-1])
        h1[i] = weights[i, :, :, :, v:early_end].sum(axis=-1).mean(axis=-1)
    return h1

h1_features = compute_h1(weights, valid_key_starts)
p_values, effect_sizes, adj_p, reject = run_per_head_test(h1_features)
print(f"H1: {reject.sum()}/336 significant (FDR); best raw p = {p_values.min():.4f}")

H1: 0/336 significant (FDR); best raw p = 0.0965


## H2 — Consistency-Head Activation

Same definition as Phase 4a. Feature: L2 norm of head output, averaged over the query window.

In [5]:
h2_features = np.linalg.norm(outputs, axis=-1).mean(axis=2)
p_values_h2, effect_sizes_h2, adj_p_h2, reject_h2 = run_per_head_test(h2_features)
print(f"H2: {reject_h2.sum()}/336 significant (FDR); best adj p = {adj_p_h2.min():.4f}")

H2: 0/336 significant (FDR); best adj p = 0.4480


## H3 — Structural Normalized Attention Entropy

For each query, entropy is calculated over real keys that are structurally
available under the causal mask. The attention distribution is renormalized
over those keys and divided by log(K_q).

In [6]:
def entropy_structural(row, valid_start, query_index):
    """Normalized entropy over real, causally available captured keys."""

    real = row[valid_start:].astype(np.float64)
    total_real_keys = len(real)

    # The query window consists of the final N_QUERIES real pre-pivot tokens.
    K_q = total_real_keys - (
        N_QUERIES - 1 - query_index
    )
    K_q = max(1, min(K_q, total_real_keys))

    available = real[:K_q]
    mass = available.sum()

    if K_q <= 1 or mass <= 0:
        return 0.0

    probabilities = available / mass
    nonzero = probabilities > 0

    raw_entropy = -np.sum(
        probabilities[nonzero]
        * np.log(probabilities[nonzero])
    )

    return raw_entropy / np.log(K_q)


def compute_h3(weights, valid_key_starts):
    h3 = np.zeros(
        (n_records, N_LAYERS, N_HEADS),
        dtype=np.float32,
    )

    for i in range(n_records):
        valid_start = int(valid_key_starts[i])

        for layer in range(N_LAYERS):
            for head in range(N_HEADS):
                entropies = [
                    entropy_structural(
                        weights[i, layer, head, query, :],
                        valid_start,
                        query,
                    )
                    for query in range(N_QUERIES)
                ]

                h3[i, layer, head] = np.mean(entropies)

    return h3


print("Computing structural normalized H3...")

h3_features = compute_h3(
    weights,
    valid_key_starts,
)

p_values_h3, effect_sizes_h3, adj_p_h3, reject_h3 = (
    run_per_head_test(h3_features)
)

pct_below_05 = float(
    (p_values_h3 < 0.05).mean()
)

print(
    f"H3: {reject_h3.sum()}/336 significant (FDR); "
    f"best raw p={p_values_h3.min():.4f}; "
    f"best adjusted p={adj_p_h3.min():.4f}; "
    f"raw p<0.05={pct_below_05:.1%}"
)

assert h3_features.shape == (
    127,
    N_LAYERS,
    N_HEADS,
)
assert np.isfinite(h3_features).all()
assert 0.07 < pct_below_05 < 0.14

Computing structural normalized H3...
H3: 0/336 significant (FDR); best raw p=0.0020; best adjusted p=0.4760; raw p<0.05=10.1%


## H4 — Late/Early Layer Magnitude Ratio

Same definition as Phase 4a. This is the hypothesis closest to significance at n=75
(p=0.057) — watch whether it weakens here, as the paper's central pattern predicts.

In [7]:
early_mag = h2_features[:, EARLY_LAYERS, :].mean(axis=(1, 2))
late_mag  = h2_features[:, LATE_LAYERS, :].mean(axis=(1, 2))
h4_features = late_mag / (early_mag + 1e-10)

diff_h4, p_h4 = permutation_test(h4_features[halluc_mask], h4_features[corr_mask], n_perm=10000)
print(f"H4: diff={diff_h4:+.4f}, p={p_h4:.4f}")

H4: diff=+0.0336, p=0.3050


## Combined classifier

Same 1,009-feature, C=0.01, instance-level 5-fold CV setup as Phase 4a. This is the
**pooled n=127** number — original and follow-up records can appear across folds
together here (Phase 4c tests what happens when that's not allowed).

In [8]:
X = np.concatenate([
    h1_features.reshape(n_records, -1),
    h2_features.reshape(n_records, -1),
    h3_features.reshape(n_records, -1),
    h4_features.reshape(n_records, 1),
], axis=1)

assert n_records == 127
assert X.shape == (127, 1009), X.shape
assert len(y) == 127
assert int(y.sum()) == 22
assert int((y == 0).sum()) == 105
assert np.isfinite(X).all()

print(f"Feature matrix: {X.shape}")

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
all_preds, all_probas = np.zeros(n_records), np.zeros(n_records)
fold_results = []

for fold_idx, (tr, te) in enumerate(skf.split(X, y)):
    sc = StandardScaler()
    Xtr, Xte = sc.fit_transform(X[tr]), sc.transform(X[te])
    clf = LogisticRegression(class_weight='balanced', max_iter=2000, C=0.01, random_state=42)
    clf.fit(Xtr, y[tr])
    preds, probas = clf.predict(Xte), clf.predict_proba(Xte)[:, 1]
    all_preds[te], all_probas[te] = preds, probas
    ba = balanced_accuracy_score(y[te], preds)
    fold_results.append(ba)
    print(f"  Fold {fold_idx}: bal_acc={ba:.3f}  (n_correction={int((y[te]==1).sum())})")

print(f"\nOverall: bal_acc={balanced_accuracy_score(y, all_preds):.3f}, "
      f"F1={f1_score(y, all_preds, zero_division=0):.3f}, "
      f"ROC-AUC={roc_auc_score(y, all_probas):.3f}")
print(f"Fold mean±SD: {np.mean(fold_results):.3f} ± {np.std(fold_results):.3f}")

Feature matrix: (127, 1009)
  Fold 0: bal_acc=0.381  (n_correction=5)
  Fold 1: bal_acc=0.652  (n_correction=5)
  Fold 2: bal_acc=0.607  (n_correction=4)
  Fold 3: bal_acc=0.429  (n_correction=4)
  Fold 4: bal_acc=0.506  (n_correction=4)

Overall: bal_acc=0.514, F1=0.208, ROC-AUC=0.579
Fold mean±SD: 0.515 ± 0.103


## Cross-domain ablation

Same as Phase 4a. Compare the math result here against Phase 4a's — this is the
paper's sharpest single illustration of non-replication.

In [9]:
print("Cross-domain ablation (train on 2 categories, test on 3rd):")
for held_out in ['math', 'logic', 'misdirection']:
    test_mask = categories == held_out
    train_mask = ~test_mask
    sc = StandardScaler()
    Xtr, Xte = sc.fit_transform(X[train_mask]), sc.transform(X[test_mask])
    clf = LogisticRegression(class_weight='balanced', max_iter=2000, C=0.01, random_state=42)
    clf.fit(Xtr, y[train_mask])
    preds, probas = clf.predict(Xte), clf.predict_proba(Xte)[:, 1]
    ba = balanced_accuracy_score(y[test_mask], preds)
    auc = roc_auc_score(y[test_mask], probas)
    print(f"  {held_out}: n={test_mask.sum()}, bal_acc={ba:.3f}, AUC={auc:.3f}")

Cross-domain ablation (train on 2 categories, test on 3rd):
  math: n=36, bal_acc=0.318, AUC=0.222
  logic: n=24, bal_acc=0.500, AUC=0.412
  misdirection: n=67, bal_acc=0.488, AUC=0.421


## Save results

In [10]:
results = {
    'n_records': n_records,
    'h1': {'n_sig': int(reject.sum()), 'best_adj_p': float(adj_p.min())},
    'h2': {'n_sig': int(reject_h2.sum()), 'best_adj_p': float(adj_p_h2.min())},
    'h3': {
    'definition': 'structural_normalized_entropy',
    'n_sig': int(reject_h3.sum()),
    'best_raw_p': float(p_values_h3.min()),
    'best_adj_p': float(adj_p_h3.min()),
    'raw_p_below_05_rate': float(
        (p_values_h3 < 0.05).mean()
    ),},
    'h4': {'p_value': float(p_h4), 'diff': float(diff_h4)},
    'classifier': {
        'balanced_accuracy': float(balanced_accuracy_score(y, all_preds)),
        'f1': float(f1_score(y, all_preds, zero_division=0)),
        'roc_auc': float(roc_auc_score(y, all_probas)),
    },
}
with open(f'{OUTPUT_DIR}/phase4b_n127_results.json', 'w') as f:
    json.dump(results, f, indent=2)
print(f"Saved to {OUTPUT_DIR}/phase4b_n127_results.json")

Saved to /kaggle/working/pivot_research/phase4b_n127_results.json
